# zero-grad-set-none — faded example 2: Order zero_grad after step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `zero-grad-set-none`. Running the beacon reports progress on the `PyTorch: zero_grad` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: zero_grad` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`zero-grad-set-none`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "zero-grad-set-none"
DD_SUBTOPIC = "PyTorch: zero_grad"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

The grad-clear must come AFTER `opt.step()`. Placing it between `backward` and `step` wipes the gradients before the optimizer uses them, freezing the parameters. Correct order: backward, step, then clear.

## Faded exercise 2

### Fix the step ordering

Implement `fixed_step(model, opt, x, y, loss_fn)` returning the pre-update loss value. Forward, loss, and backward are done; you must call step and then clear grads in the right order. Complete the line that performs the optimizer update (it must come BEFORE clearing grads).

**Fill in:** the opt.step() call placed before the grad-clearing loop

In [ ]:
import torch as t
import torch.nn as nn

def fixed_step(model, opt, x, y, loss_fn):
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    for p in model.parameters():
        p.grad = None
    return loss.item()

t.manual_seed(0)
m = nn.Linear(3, 1)
opt = t.optim.SGD(m.parameters(), lr=0.05)
print(fixed_step(m, opt, t.randn(8, 3), t.randn(8, 1), nn.MSELoss()))


def _test():
    t.manual_seed(0)
    m = nn.Linear(3, 1)
    opt = t.optim.SGD(m.parameters(), lr=0.05)
    x, y = t.randn(16, 3), t.randn(16, 1)
    loss_fn = nn.MSELoss()
    losses = [fixed_step(m, opt, x, y, loss_fn) for _ in range(8)]
    # parameters actually moved -> loss strictly improved overall
    assert losses[-1] < losses[0], losses
    # monotone non-increasing on this convex problem with small lr
    assert all(losses[i] >= losses[i + 1] - 1e-6 for i in range(len(losses) - 1)), losses
    # grads cleared after the final step
    assert all(p.grad is None for p in m.parameters())


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def fixed_step(model, opt, x, y, loss_fn):
    pred = model(x)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()
    for p in model.parameters():
        p.grad = None
    return loss.item()

t.manual_seed(0)
m = nn.Linear(3, 1)
opt = t.optim.SGD(m.parameters(), lr=0.05)
print(fixed_step(m, opt, t.randn(8, 3), t.randn(8, 1), nn.MSELoss()))
```
</details>